# Synthetic 2D Heatmaps — All KLIG Variants

A_x − A_y attribution diff maps across all methods on 5 toy 2D scalar functions.
Each panel shows **attribution along x₁ minus attribution along x₂** for every
grid point, revealing each method's sensitivity geometry.

| Method | Description |
|---|---|
| **IG** | Standard Integrated Gradients (zero baseline, deterministic) |
| **KLIG-Adaptive** | KLIntegratedGradients + LinearPath + per-function adaptive σ |
| **DDPath-cos/lin/quad** | Diffusion paths with three noise schedules |
| **Greedy-μ** | GreedyMuAttributor (μ-only greedy step weighting) |
| **Greedy-Jt** | GreedyJointAttributor (joint μ+lv step weighting) |
| **Greedy-Srt** | SortedDimPath (per-dim γ ranked by prior gradient magnitude) |
| **KL-IG²** | RepDescentPath (distribution-space representation descent) |

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
!git clone --branch claude/general-session-FcgoB \
    https://github.com/Shameen5375/KLIG_V1.git /content/KLIG_V1 2>/dev/null || \
    (cd /content/KLIG_V1 && git pull) 2>/dev/null
import os, sys
for _root in ['/content/KLIG_V1/infocube-main', 'infocube-main', '.']:
    if os.path.isdir(_root) and _root not in sys.path:
        sys.path.insert(0, _root)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from __future__ import annotations
import math, time, warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

from klig import (
    KLIntegratedGradients,
    GreedyMuAttributor,
    GreedyJointAttributor,
    SortedDimPath,
    DDiffusionPath,
    RepDescentPath,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)


In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
EXTENT          = 2.0
GRID_RESOLUTION = 40    # 1600 query points; bump to 60 for publication quality
HEAT_N          = 200   # function heatmap resolution

# Shared integrator / greedy settings
N_STEPS    = 24         # integration / greedy steps
N_SAMPLES  = 8          # MC samples

# Fixed sigma (for DDPath / Greedy / KL-IG²)
SIGMA_F    = 0.05

# ── Adaptive sigma — uses the core find_sigma_stop, exactly as in ImageAttributor
# ToyClassifierWrapper turns a scalar fn into a 2-class logit model so that
# find_sigma_stop can binary-search over model confidence, per query point.
from klig.image.stopping import find_sigma_stop

ADAPTIVE_TAU       = 0.95   # confidence retention threshold (core library default)
ADAPTIVE_N_SAMPLES = 32     # MC samples per bisection step
ADAPTIVE_N_ITER    = 12     # bisection iterations
ADAPTIVE_FALLBACK  = SIGMA_F  # fallback near zero-crossings where |f(x)| is tiny
ADAPTIVE_ZERO_THR  = 0.05   # |f(x)| < this → use ADAPTIVE_FALLBACK

class ToyClassifierWrapper(nn.Module):
    """Wrap a scalar 2D function as a binary logit classifier.

    Outputs (B, 2) logits [f(x)*scale, -f(x)*scale] so that find_sigma_stop
    can run its binary search over model confidence — identical to the image case.
    """
    def __init__(self, fn, scale: float = 5.0):
        super().__init__()
        self.fn    = fn
        self.scale = scale
        # dummy parameter so next(model.parameters()).device resolves correctly
        self._dev  = nn.Parameter(torch.zeros(1), requires_grad=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 2)  →  (B, 2) logits
        logit = self.fn(x) * self.scale          # (B,)
        return torch.stack([logit, -logit], dim=1)

def get_sigma_adaptive_2d(clf, fn, x_query: torch.Tensor) -> float:
    """Per-point adaptive sigma via find_sigma_stop (same algorithm as core library).

    target = 0 if f(x) >= 0 else 1  (positive / negative class).
    Falls back to ADAPTIVE_FALLBACK when |f(x)| < ADAPTIVE_ZERO_THR (zero-crossings).
    sigma_hi=1.0 caps the search: domain is [-2,2], larger sigma escapes support.
    """
    with torch.no_grad():
        f_val = fn(x_query.unsqueeze(0)).item()
    if abs(f_val) < ADAPTIVE_ZERO_THR:
        return ADAPTIVE_FALLBACK
    target = 0 if f_val >= 0 else 1
    return find_sigma_stop(
        clf, x_query, target=target, tau=ADAPTIVE_TAU,
        n_samples=ADAPTIVE_N_SAMPLES, n_iter=ADAPTIVE_N_ITER,
        sigma_hi=1.0,
    )

# SortedDimPath (Greedy-Sorted)
GAMMA_LO   = 0.25
GAMMA_HI   = 4.0
GAMMA_N_MC = 64

# KL-IG² (RepDescentPath)
IG2_T         = 15
IG2_LR_MU    = 0.05
IG2_LR_LV    = 0.10
IG2_N_MC     = 4
IG2_LOSS_STOP = 1e-3
IG2_LV_FLOOR  = 2 * math.log(1 / 20)
IG2_LV_CEIL   = 2.0

In [ ]:
# ── Toy 2D functions (verbatim from synthetic_funcs_heatmap_attributions.py) ─
class XOR(nn.Module):
    def __init__(self, s=5.0): super().__init__(); self.s = s
    def forward(self, x):
        return torch.tanh(self.s*x[...,0]) * torch.tanh(self.s*x[...,1])

class Checkerboard(nn.Module):
    def __init__(self, period=1.0, s=15.0):
        super().__init__(); self.period=period; self.s=s
    def forward(self, x):
        u = torch.cos(math.pi*x[...,0]/self.period)
        v = torch.cos(math.pi*x[...,1]/self.period)
        return torch.tanh(self.s * u * v)

class DiagonalCheckerboard(nn.Module):
    def __init__(self, period=1.0, s=15.0):
        super().__init__(); self.period=period; self.s=s
    def forward(self, x):
        u = (x[...,0]+x[...,1])/math.sqrt(2)
        v = (x[...,0]-x[...,1])/math.sqrt(2)
        cu = torch.cos(math.pi*u/self.period)
        cv = torch.cos(math.pi*v/self.period)
        return torch.tanh(self.s * cu * cv)

class RadialRings(nn.Module):
    def __init__(self, k=1.0, sigma=1.5):
        super().__init__(); self.k=k; self.sigma=sigma
    def forward(self, x):
        r = torch.norm(x, dim=-1)
        env = torch.exp(-r**2/(2*self.sigma**2))
        return env * torch.cos(2*math.pi*self.k*r)

class FlatFarFieldBumps(nn.Module):
    def __init__(self,
                 centers=((0.15,-0.10),(-0.12,0.18),(0.05,0.05)),
                 amps=(1.0,-0.85,0.70), sigma=0.12):
        super().__init__()
        self.register_buffer('centers', torch.tensor(centers, dtype=torch.float32))
        self.register_buffer('amps',    torch.tensor(amps,    dtype=torch.float32))
        self.sigma = sigma
    def forward(self, x):
        diff = x.unsqueeze(-2) - self.centers
        r2   = (diff**2).sum(-1)
        return (torch.exp(-r2/(2*self.sigma**2)) * self.amps).sum(-1)

FUNCS = {
    'xor':            XOR(5.0).to(DEVICE),
    'checkerboard':   Checkerboard(1.0, 15.0).to(DEVICE),
    'diagonal_ckb':   DiagonalCheckerboard(1.0, 15.0).to(DEVICE),
    'radial':         RadialRings(1.0, 1.5).to(DEVICE),
    'flat_far_field': FlatFarFieldBumps().to(DEVICE),
}
FUNC_NAMES = list(FUNCS)
print('functions:', FUNC_NAMES)

In [ ]:
# ── Grid helpers ─────────────────────────────────────────────────────────────
def grid_points(n, extent=EXTENT):
    xs = np.linspace(-extent, extent, n)
    X, Y = np.meshgrid(xs, xs, indexing='xy')
    return np.stack([X.flatten(), Y.flatten()], axis=1).astype(np.float32)

def evaluate_heat(fn, n=HEAT_N, extent=EXTENT):
    xs = torch.linspace(-extent, extent, n, device=DEVICE)
    X, Y = torch.meshgrid(xs, xs, indexing='xy')
    pts = torch.stack([X.flatten(), Y.flatten()], dim=1)
    with torch.no_grad():
        return fn(pts).cpu().numpy().reshape(n, n)

In [ ]:
# ── Scalar objective for 2D nn.Modules ───────────────────────────────────────
# target = lambda y: y  passes fn output (shape: n_samples) directly;
# KLIntegratedGradients wraps it as target(fn(x_samp)).mean() → scalar.
_SCALAR_TARGET = lambda y: y

# ── ∇f (vanilla gradient) ────────────────────────────────────────────────────
def gradient_field(fn, points):
    pts = torch.tensor(points, device=DEVICE).requires_grad_(True)
    y   = fn(pts)
    g   = torch.autograd.grad(y.sum(), pts)[0]
    return g.detach().cpu().numpy()

# ── Standard IG (zero baseline, deterministic trapezoid) ─────────────────────
def integrated_gradients(fn, points, n_steps=64):
    pts   = torch.tensor(points, device=DEVICE)
    B, D  = pts.shape
    base  = torch.zeros(D, device=DEVICE)
    delta = pts - base
    attr  = torch.zeros(B, D, device=DEVICE)
    for k in range(n_steps):
        alpha = (k + 0.5) / n_steps
        xa = (base + alpha * delta).requires_grad_(True)
        g  = torch.autograd.grad(fn(xa).sum(), xa)[0]
        attr += g.detach() * delta / n_steps
    return attr.cpu().numpy()


In [ ]:
# ── KLIG-Adaptive ─────────────────────────────────────────────────────────────
# Calls find_sigma_stop per query point via ToyClassifierWrapper —
# identical to ImageAttributor(model, sigma_final=find_sigma_stop(...)).
def klig_adaptive_field(fn, points, clf):
    attr_ = np.zeros_like(points)
    for i in range(len(points)):
        x     = torch.tensor(points[i], device=DEVICE)
        sigma = get_sigma_adaptive_2d(clf, fn, x)
        ig    = KLIntegratedGradients(fn, n_steps=N_STEPS, n_samples=N_SAMPLES,
                                       sigma_final=sigma, device=DEVICE)
        r     = ig.attribute(x, target=_SCALAR_TARGET)
        attr_[i] = r.attr.detach().cpu().numpy()
    return attr_

# ── DDPath variants ───────────────────────────────────────────────────────────
def _ddpath_field(fn, points, schedule):
    attr_ = np.zeros_like(points)
    ig    = KLIntegratedGradients(fn, n_steps=N_STEPS, n_samples=N_SAMPLES,
                                   sigma_final=SIGMA_F,
                                   path=DDiffusionPath(schedule=schedule),
                                   device=DEVICE)
    for i in range(len(points)):
        x = torch.tensor(points[i], device=DEVICE)
        r = ig.attribute(x, target=_SCALAR_TARGET)
        attr_[i] = r.attr.detach().cpu().numpy()
    return attr_

def ddpath_cos_field(fn, points):  return _ddpath_field(fn, points, 'cosine')
def ddpath_lin_field(fn, points):  return _ddpath_field(fn, points, 'linear')
def ddpath_quad_field(fn, points): return _ddpath_field(fn, points, 'quadratic')

# ── Greedy-μ ──────────────────────────────────────────────────────────────────
def greedy_mu_field(fn, points):
    attr_ = np.zeros_like(points)
    g = GreedyMuAttributor(fn, n_steps=N_STEPS, n_samples=N_SAMPLES,
                            sigma_final=SIGMA_F, device=DEVICE)
    for i in range(len(points)):
        x = torch.tensor(points[i], device=DEVICE)
        r = g.attribute(x, target=_SCALAR_TARGET)
        attr_[i] = r.attr.detach().cpu().numpy()
    return attr_

# ── Greedy-Joint ──────────────────────────────────────────────────────────────
def greedy_joint_field(fn, points):
    attr_ = np.zeros_like(points)
    g = GreedyJointAttributor(fn, n_steps=N_STEPS, n_samples=N_SAMPLES,
                               sigma_final=SIGMA_F, device=DEVICE)
    for i in range(len(points)):
        x = torch.tensor(points[i], device=DEVICE)
        r = g.attribute(x, target=_SCALAR_TARGET)
        attr_[i] = r.attr.detach().cpu().numpy()
    return attr_

# ── Greedy-Sorted (SortedDimPath) ─────────────────────────────────────────────
def build_sorted_dim_path_2d(fn):
    with torch.no_grad():
        eps = torch.randn(GAMMA_N_MC, 2, device=DEVICE)
    eps.requires_grad_(True)
    y = fn(eps)
    g = torch.autograd.grad(y.sum(), eps)[0]
    with torch.no_grad():
        g_mag = g.abs().mean(0)
    order  = g_mag.argsort(descending=True)
    gamma  = torch.empty(2, device=DEVICE)
    gamma[order] = torch.tensor([GAMMA_LO, GAMMA_HI], device=DEVICE)
    return SortedDimPath(gamma)

def greedy_sorted_field(fn, points):
    attr_ = np.zeros_like(points)
    path  = build_sorted_dim_path_2d(fn)
    ig    = KLIntegratedGradients(fn, n_steps=N_STEPS, n_samples=N_SAMPLES,
                                   sigma_final=SIGMA_F, path=path, device=DEVICE)
    for i in range(len(points)):
        x = torch.tensor(points[i], device=DEVICE)
        r = ig.attribute(x, target=_SCALAR_TARGET)
        attr_[i] = r.attr.detach().cpu().numpy()
    return attr_

# ── KL-IG² (RepDescentPath) ───────────────────────────────────────────────────
def klig2_field(fn, points):
    attr_  = np.zeros_like(points)
    x_cf   = torch.zeros(2, device=DEVICE)
    phi    = lambda x: x

    for i in range(len(points)):
        x   = torch.tensor(points[i], device=DEVICE)
        path = RepDescentPath(
            phi=phi, x_cf=x_cf,
            T=IG2_T, lr_mu=IG2_LR_MU, lr_lv=IG2_LR_LV,
            n_mc=IG2_N_MC, loss_stop=IG2_LOSS_STOP,
            lv_floor=IG2_LV_FLOOR, lv_ceil=IG2_LV_CEIL,
            mu_min=-EXTENT, mu_max=EXTENT, clamp_samples=True,
        )
        ig  = KLIntegratedGradients(fn, n_steps=N_STEPS, n_samples=N_SAMPLES,
                                    sigma_final=SIGMA_F, path=path, device=DEVICE)
        r   = ig.attribute(x, target=_SCALAR_TARGET)
        attr_[i] = r.attr.detach().cpu().numpy()
    return attr_

In [ ]:
# ── Compute attributions for every function × method ────────────────────────
points = grid_points(GRID_RESOLUTION)
data   = {}

# Build a ToyClassifierWrapper per function for KLIG-Adaptive's find_sigma_stop
clf_dict = {fname: ToyClassifierWrapper(fn).to(DEVICE) for fname, fn in FUNCS.items()}

for fname, fn in FUNCS.items():
    clf = clf_dict[fname]
    METHODS = [
        ('IG',           'ig',           lambda fn, pts: integrated_gradients(fn, pts)),
        ('KLIG-Adapt',   'klig_adapt',   lambda fn, pts, c=clf: klig_adaptive_field(fn, pts, c)),
        ('DDPath-cos',   'ddpath_cos',   lambda fn, pts: ddpath_cos_field(fn, pts)),
        ('DDPath-lin',   'ddpath_lin',   lambda fn, pts: ddpath_lin_field(fn, pts)),
        ('DDPath-quad',  'ddpath_quad',  lambda fn, pts: ddpath_quad_field(fn, pts)),
        ('Greedy-μ',     'greedy_mu',    lambda fn, pts: greedy_mu_field(fn, pts)),
        ('Greedy-Jt',    'greedy_jt',    lambda fn, pts: greedy_joint_field(fn, pts)),
        ('Greedy-Srt',   'greedy_srt',   lambda fn, pts: greedy_sorted_field(fn, pts)),
        ('KL-IG²',       'klig2',        lambda fn, pts: klig2_field(fn, pts)),
    ]
    if f'{fname}_heat' not in data:
        data[f'{fname}_heat'] = evaluate_heat(fn)
    print(f'[{fname}]', end='  ')
    for mname, key, method_fn in METHODS:
        t0 = time.time()
        data[f'{fname}_{key}'] = method_fn(fn, points)
        print(f'{mname}={time.time()-t0:.1f}s', end='  ')
    print()

# Store METHODS list for the render cell
METHODS_LIST = [
    ('IG',          'ig'),
    ('KLIG-Adapt',  'klig_adapt'),
    ('DDPath-cos',  'ddpath_cos'),
    ('DDPath-lin',  'ddpath_lin'),
    ('DDPath-quad', 'ddpath_quad'),
    ('Greedy-μ',    'greedy_mu'),
    ('Greedy-Jt',   'greedy_jt'),
    ('Greedy-Srt',  'greedy_srt'),
    ('KL-IG²',      'klig2'),
]
print('\nDone.')

In [ ]:
# ── Render: rows = functions, cols = f + each method's (A_x − A_y) ───────────
from mpl_toolkits.axes_grid1 import make_axes_locatable

CLIP_PCT  = 98.0   # clip attribution maps at 98th percentile — removes outlier spikes
n         = GRID_RESOLUTION

METHOD_COLORS = {
    'IG':          '#555555',
    'KLIG-Adapt':  '#2d6a2d',
    'DDPath-cos':  '#1f77b4',
    'DDPath-lin':  '#4a90d9',
    'DDPath-quad': '#d62728',
    'Greedy-μ':    '#ff7f0e',
    'Greedy-Jt':   '#e377c2',
    'Greedy-Srt':  '#9467bd',
    'KL-IG²':      '#e41a1c',
}

nrows = len(FUNC_NAMES)
ncols = 1 + len(METHODS_LIST)

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(2.2 * ncols, 2.4 * nrows),
    facecolor='white',
    gridspec_kw={'wspace': 0.04, 'hspace': 0.12},
)
if nrows == 1: axes = axes.reshape(1, ncols)

FUNC_LABELS = {
    'xor':           'XOR',
    'checkerboard':  'Checkerboard',
    'diagonal_ckb':  'Diagonal\nCheckerboard',
    'radial':        'Radial Rings',
    'flat_far_field':'Flat Far-Field\nBumps',
}

for ri, fname in enumerate(FUNC_NAMES):
    # ── col 0: function heatmap ──────────────────────────────────────────────
    heat = data[f'{fname}_heat']
    vmax = max(1e-6, float(np.abs(heat).max()))
    ax0  = axes[ri, 0]
    im0  = ax0.imshow(heat, extent=[-EXTENT, EXTENT, -EXTENT, EXTENT],
                       origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                       interpolation='bilinear')
    ax0.set_xticks([]); ax0.set_yticks([])
    ax0.set_ylabel(FUNC_LABELS.get(fname, fname), fontsize=9,
                    rotation=90, labelpad=4, va='center')
    if ri == 0:
        ax0.set_title('f(x₁, x₂)', fontsize=9, fontweight='bold', pad=5)
    # thin colorbar
    div  = make_axes_locatable(ax0)
    cax  = div.append_axes('right', size='8%', pad=0.04)
    cb   = plt.colorbar(im0, cax=cax)
    cb.ax.tick_params(labelsize=6)

    # ── cols 1–N: A_x − A_y maps ─────────────────────────────────────────────
    for ci, (mname, key) in enumerate(METHODS_LIST, start=1):
        attrs = data[f'{fname}_{key}']
        diff  = (attrs[:, 0] - attrs[:, 1]).reshape(n, n)
        # 98th-percentile clipping — suppresses spike outliers cleanly
        ref   = float(np.percentile(np.abs(diff), CLIP_PCT))
        ref   = max(ref, 1e-3)
        ax    = axes[ri, ci]
        im    = ax.imshow(diff, extent=[-EXTENT, EXTENT, -EXTENT, EXTENT],
                           origin='lower', cmap='PuOr',
                           vmin=-ref, vmax=ref, interpolation='bilinear')
        ax.set_xticks([]); ax.set_yticks([])
        # remove all spines for a clean look
        for sp in ax.spines.values(): sp.set_visible(False)
        if ri == 0:
            col = METHOD_COLORS.get(mname, 'black')
            ax.set_title(mname, fontsize=8.5, fontweight='bold',
                          color=col, pad=5)

# shared x-axis label at the bottom
fig.text(0.55, -0.01, 'x₁', ha='center', va='bottom', fontsize=10)

fig.suptitle(
    r'$A_{x_1} - A_{x_2}$  attribution difference maps'
    f'  ({n}×{n} grid, EXTENT={EXTENT})\n'
    'Orange → prefers x₁   |   Purple → prefers x₂',
    fontsize=11, fontweight='bold', y=1.01)

plt.savefig('synthetic_heatmap_all_methods.png', dpi=180, bbox_inches='tight',
            facecolor='white')
plt.show()

## Path Attribution Change Curves

For each method, how much attribution is accumulated at each step along the path?

At integration step k (path position αₖ):

    Δattr_k = g_μ(αₖ) · dμ(αₖ) + g_lv(αₖ) · dlv(αₖ)  (KLIG-family)
    Δattr_k = ∇f(αₖ · x) · x / N_STEPS                  (IG)

where `dμ/dlv` are the path derivatives at αₖ.

The curve `|Δattr_k|` vs αₖ reveals **when** each method concentrates its
attribution signal along the path.  Methods that peak early use the baseline
region; methods that peak near α=1 emphasise the explicand neighbourhood.

Averaged over N_SAMPLE_PTS random query points and all 5 functions.
Two panels: **absolute** L1 magnitude (log scale) and **normalised** (fraction
of each method's total attribution change).

In [ ]:
# ── Per-step attribution change helpers ──────────────────────────────────────
from klig.core.path import LinearPath
import math

def _klig_step_change(fn, x, path, sigma=SIGMA_F, n_steps=N_STEPS, n_samples=N_SAMPLES):
    """Per-step |Δattr_k| for any DistributionPath. Returns np.ndarray (n_steps,)."""
    mu_f   = x.detach()
    lv_f   = torch.full_like(mu_f, 2 * math.log(sigma))
    alphas = torch.linspace(0.5/n_steps, 1 - 0.5/n_steps, n_steps, device=DEVICE)
    change = np.zeros(n_steps, dtype=np.float32)
    for k, t in enumerate(alphas.tolist()):
        mu_t,  lv_t  = path.at(t,  mu_f, lv_f)
        dmu_t, dlv_t = path.derivatives(t, mu_f, lv_f)
        mu_p = mu_t.detach().requires_grad_(True)
        lv_p = lv_t.detach().requires_grad_(True)
        eps    = torch.randn(n_samples, *x.shape, device=DEVICE)
        x_samp = mu_p.unsqueeze(0) + (0.5 * lv_p).exp().unsqueeze(0) * eps
        obj    = _SCALAR_TARGET(fn(x_samp)).mean()
        g_mu, g_lv = torch.autograd.grad(obj, [mu_p, lv_p])
        change[k] = float(((g_mu * dmu_t + g_lv * dlv_t) / n_steps).abs().sum().item())
    return change

def _ig_step(fn, x, n_steps=N_STEPS):
    alphas = np.linspace(0.5/n_steps, 1 - 0.5/n_steps, n_steps)
    change = np.zeros(n_steps, dtype=np.float32)
    for k, alpha in enumerate(alphas):
        xa = (alpha * x).unsqueeze(0).requires_grad_(True)
        g  = torch.autograd.grad(fn(xa).sum(), xa)[0]
        change[k] = float((g.detach()[0] * x / n_steps).abs().sum().item())
    return change

def _greedy_mu_step(fn, x):
    r = GreedyMuAttributor(fn, n_steps=N_STEPS, n_samples=N_SAMPLES,
                            sigma_final=SIGMA_F, device=DEVICE
                           ).attribute(x, target=_SCALAR_TARGET)
    return np.array(r.step_grad_signal, dtype=np.float32)

def _greedy_jt_step(fn, x):
    r = GreedyJointAttributor(fn, n_steps=N_STEPS, n_samples=N_SAMPLES,
                               sigma_final=SIGMA_F, device=DEVICE
                              ).attribute(x, target=_SCALAR_TARGET)
    return np.array(r.step_grad_signal, dtype=np.float32)

def _greedy_srt_step(fn, x):
    return _klig_step_change(fn, x, build_sorted_dim_path_2d(fn))

def _klig2_step(fn, x):
    path = RepDescentPath(
        phi=lambda xb: xb, x_cf=torch.zeros(2, device=DEVICE),
        T=IG2_T, lr_mu=IG2_LR_MU, lr_lv=IG2_LR_LV,
        n_mc=IG2_N_MC, loss_stop=IG2_LOSS_STOP,
        lv_floor=IG2_LV_FLOOR, lv_ceil=IG2_LV_CEIL,
        mu_min=-EXTENT, mu_max=EXTENT, clamp_samples=True)
    return _klig_step_change(fn, x, path)

STEP_FNS = {
    'IG':           lambda fn, x, s=None: _ig_step(fn, x),
    'KLIG-Adapt':   lambda fn, x, s=None: _klig_step_change(fn, x, LinearPath(), sigma=s if s is not None else SIGMA_F),
    'DDPath-cos':   lambda fn, x, s=None: _klig_step_change(fn, x, DDiffusionPath('cosine')),
    'DDPath-lin':   lambda fn, x, s=None: _klig_step_change(fn, x, DDiffusionPath('linear')),
    'DDPath-quad':  lambda fn, x, s=None: _klig_step_change(fn, x, DDiffusionPath('quadratic')),
    'Greedy-μ':     lambda fn, x, s=None: _greedy_mu_step(fn, x),
    'Greedy-Jt':    lambda fn, x, s=None: _greedy_jt_step(fn, x),
    'Greedy-Srt':   lambda fn, x, s=None: _greedy_srt_step(fn, x),
    'KL-IG²':       lambda fn, x, s=None: _klig2_step(fn, x),
}

CURVE_COLORS = {
    'IG':          '#555555',
    'KLIG-Adapt':  '#2d6a2d',
    'DDPath-cos':  '#1f77b4',
    'DDPath-lin':  '#4a90d9',
    'DDPath-quad': '#d62728',
    'Greedy-μ':    '#ff7f0e',
    'Greedy-Jt':   '#e377c2',
    'Greedy-Srt':  '#9467bd',
    'KL-IG²':      '#e41a1c',
}
CURVE_LS = {
    'IG':          '-',
    'KLIG-Adapt':  '-',
    'DDPath-cos':  '--',
    'DDPath-lin':  '--',
    'DDPath-quad': '--',
    'Greedy-μ':    '-.',
    'Greedy-Jt':   '-.',
    'Greedy-Srt':  '-.',
    'KL-IG²':      ':',
}

# ── Sample random query points ────────────────────────────────────────────────
N_SAMPLE_PTS = 30
rng_curve    = np.random.default_rng(42)
all_pts      = grid_points(GRID_RESOLUTION)
sample_idx   = rng_curve.choice(len(all_pts), N_SAMPLE_PTS, replace=False)
sample_pts   = all_pts[sample_idx]

# ── Compute curves[fname][mname] = (N_SAMPLE_PTS, N_STEPS) ──────────────────
print(f'Computing per-step changes: {len(FUNCS)} fns × {len(STEP_FNS)} methods '
      f'× {N_SAMPLE_PTS} pts ...')
curves = {fname: {mname: np.zeros((N_SAMPLE_PTS, N_STEPS), dtype=np.float32)
                  for mname in STEP_FNS}
          for fname in FUNC_NAMES}

for fname, fn in FUNCS.items():
    clf = clf_dict[fname]
    print(f'  [{fname}]', end='  ')
    for mname, step_fn in STEP_FNS.items():
        for pi, pt in enumerate(sample_pts):
            x = torch.tensor(pt, device=DEVICE)
            # KLIG-Adaptive: per-point sigma via find_sigma_stop (core library)
            s = get_sigma_adaptive_2d(clf, fn, x) if mname == 'KLIG-Adapt' else None
            curves[fname][mname][pi] = step_fn(fn, x, s)
        print(mname, end=' ')
    print()
print('Done.')

In [ ]:
# ── Plot: per-function columns + per-row absolute/normalised ──────────────────
alphas_ref = np.linspace(0.5/N_STEPS, 1 - 0.5/N_STEPS, N_STEPS)

nrows, ncols = 2, len(FUNC_NAMES)
fig, axes = plt.subplots(nrows, ncols,
                          figsize=(4.5 * ncols, 4.0 * nrows),
                          facecolor='white', sharex=True)

for ci, fname in enumerate(FUNC_NAMES):
    ax_abs  = axes[0, ci]
    ax_norm = axes[1, ci]

    for mname in STEP_FNS:
        arr  = curves[fname][mname]          # (N_SAMPLE_PTS, N_STEPS)
        mean = arr.mean(axis=0)
        sem  = arr.std(axis=0) / N_SAMPLE_PTS ** 0.5
        color, ls = CURVE_COLORS[mname], CURVE_LS[mname]

        # absolute (top row, log-scale)
        ax_abs.plot(alphas_ref, np.maximum(mean, 1e-9),
                    color=color, ls=ls, lw=1.8, label=mname, zorder=3)
        ax_abs.fill_between(alphas_ref,
                             np.maximum(mean - sem, 1e-9),
                             np.maximum(mean + sem, 1e-9),
                             color=color, alpha=0.15, zorder=2)

        # normalised (bottom row)
        totals = arr.sum(axis=1, keepdims=True) + 1e-12
        norm   = (arr / totals).mean(axis=0)
        norm_sem = (arr / totals).std(axis=0) / N_SAMPLE_PTS ** 0.5
        ax_norm.plot(alphas_ref, norm,
                     color=color, ls=ls, lw=1.8, zorder=3)
        ax_norm.fill_between(alphas_ref, norm - norm_sem, norm + norm_sem,
                              color=color, alpha=0.15, zorder=2)

    ax_abs.set_yscale('log')
    ax_abs.set_title(fname, fontsize=11, fontweight='bold')
    ax_abs.set_ylabel(r'$|\Delta attr_k|$  (log)', fontsize=9)
    ax_abs.grid(True, which='both', alpha=0.25, linestyle='--')

    ax_norm.set_xlabel(r'Path position $\alpha$', fontsize=9)
    ax_norm.set_ylabel('Fraction of total |Δattr|', fontsize=9)
    ax_norm.set_xlim(0, 1); ax_norm.set_ylim(bottom=0)
    ax_norm.grid(True, alpha=0.25, linestyle='--')

# single legend from first panel
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, 1.02), framealpha=0.95)

fig.suptitle(
    f'Path Attribution Change Curves  '
    f'(mean ± SEM over {N_SAMPLE_PTS} random query pts, {len(FUNCS)} functions)\n'
    'Top: absolute |Δattr_k| per step (log scale)   '
    'Bottom: normalised — where each method concentrates attribution',
    fontsize=11, y=1.05)
plt.tight_layout()
plt.savefig('synthetic_path_change_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary: overall normalised peak position per method ──────────────────────
print(f'\n{"Method":<14}  {"peak α (overall)":>16}  {"Gini (concentration)":>20}')
print('-' * 55)
for mname in STEP_FNS:
    # pool across all functions and sample points
    all_arr = np.concatenate([curves[f][mname] for f in FUNC_NAMES], axis=0)
    mean    = all_arr.mean(0)
    norm    = mean / (mean.sum() + 1e-12)
    peak_a  = float(alphas_ref[norm.argmax()])
    v = np.sort(mean.ravel()); n = len(v)
    gini = float((2*np.arange(1,n+1) - n - 1) @ v / (n * v.sum() + 1e-12))
    print(f'{mname:<14}  {peak_a:>16.3f}  {gini:>20.3f}')

## Reading the panels

Each non-`f` column shows `A_x − A_y` for one method. Orange = method credits
the **x-feature** more than y; purple = the opposite. The five rows expose
different failure modes:

- **xor** — saddle at origin. Methods that respect the bilinear interaction
  should show four sign-alternating quadrants. IG with a zero baseline tends
  to look bland here because the straight-line path averages the interaction
  away.
- **checkerboard / diagonal_ckb** — diagonal_ckb is only separable in the
  rotated `(x+y, x−y)` basis, so axis-aligned methods (IG, SHAP-style) will
  smear blame across both features symmetrically; KL-path / greedy methods
  do somewhat better.
- **radial** — by symmetry `A_x − A_y` should be zero on the diagonals
  and antisymmetric across them. Any method that violates this on average
  is leaking baseline geometry into the attribution.
- **flat_far_field** — far from the bumps the function is flat, so the
  ground-truth A_x − A_y is ≈0 there. Any non-trivial colour in the far
  field is the IG "shadow effect" (a straight path through the bump
  cluster). KL-IG² and DDPath should be visibly cleaner there.

To sharpen the maps, bump `GRID_RESOLUTION` to 60–100 (longer wall-time)
and/or increase `N_STEPS`, `N_SAMPLES`.

## Path geometry taxonomy — small multiples + unified overlay

Each method's trajectory is plotted in `(μ-displacement, logvar-displacement)` space,
both axes normalised per-method to `[0, 1]`. A faint dashed **linear reference** appears
on every panel.

**Three-family hypothesis:** path geometry predicts failure mode.

| Family | Members | Geometry | Fails when |
|---|---|---|---|
| **Data-agnostic schedules** | KL-IG linear, DDPath (cosine/linear/quadratic), PowerPath (p=2/p=0.5) | Fixed parametric curve; ignores `f` entirely | Path transits high-gradient regions unrelated to the target class (IG shadow effect) |
| **Classifier-informed descent** | KLIG2 pixel-space, KL-Descent (max-ent / sharp / fuzzy) | Curve bent by `∇KL` in pixel space, or by an explicit counterfactual `(μ_cf, lv_cf)` | Counterfactual mis-specified → path misses the semantic contrast |
| **Self-referential greedy** | GreedyMu, GreedyJoint | Staircase weighted by `|∂f/∂μ|` (or `|∂f/∂logvar|`) at the current waypoint | Early steps over-commit to dims that turn out irrelevant |

**Phase (a)** below uses synthetic curves with the qualitatively correct shape for
each method. **Phase (b)** at the bottom shows how to swap in real waypoints.

In [ ]:
# ── Synthetic trajectory functions (Phase a) ─────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

S = np.linspace(0.0, 1.0, 300)

def _norm(x):
    x = np.asarray(x, dtype=float)
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo + 1e-12)

# ── Data-agnostic schedules ──────────────────────────────────────────────────
def traj_linear(s=S):
    return _norm(s), _norm(s)

def traj_dd_cosine(s=S):
    ab = np.sin(np.pi * s / 2) ** 2      # ᾱ = sin²(πs/2)
    mu = np.sqrt(ab + 1e-6)              # μ(s) ∝ √ᾱ  — slow start, fast finish
    lv = np.log(1 - ab + 1e-6)           # logvar → −∞ at s=1
    return _norm(mu), _norm(-lv)          # flip sign: displacement increases toward s=1

def traj_dd_linear(s=S):
    ab = s
    mu = np.sqrt(ab + 1e-6)
    lv = np.log(1 - ab + 1e-6)
    return _norm(mu), _norm(-lv)

def traj_dd_quadratic(s=S):
    ab = s ** 2
    mu = np.sqrt(ab + 1e-6)
    lv = np.log(1 - ab + 1e-6)
    return _norm(mu), _norm(-lv)

def traj_power_p2(s=S):
    return _norm(s ** 2), _norm(s)        # μ lags behind uniform lv

def traj_power_p05(s=S):
    return _norm(np.sqrt(s)), _norm(s)    # μ leads ahead of uniform lv

# ── Classifier-informed descent ──────────────────────────────────────────────
def traj_klig2_pixel(s=S):
    # Pixel-space only: lv axis is undefined → rendered at lv=0 (flat horizontal)
    return _norm(1 - (1 - s) ** 1.4), np.zeros_like(s)

def traj_kld_maxent(s=S):
    # μ_cf=0, lv_cf=0  → exponential lerp on both axes at the same rate
    k = 3.5
    mu = 1 - np.exp(-k * s)
    lv = 1 - np.exp(-k * s)
    return _norm(mu), _norm(lv)

def traj_kld_sharp(s=S):
    # lv_cf ≈ −11 (tiny σ) → logvar gap is enormous; μ closes faster than lv
    mu = 1 - np.exp(-2.8 * s)
    lv = 1 - np.exp(-5.5 * s) ** 1.3     # lv displacement lags, then catches up
    return _norm(mu), _norm(lv)

def traj_kld_fuzzy(s=S):
    # lv_cf=0, μ_cf = other-class image → like max-ent but μ has a longer path
    k = 3.0
    mu = 1 - np.exp(-k * 0.85 * s)       # slightly slower than max-ent
    lv = 1 - np.exp(-k * s)
    return _norm(mu), _norm(lv)

# ── Self-referential greedy ──────────────────────────────────────────────────
def _greedy_mu_disp(s, n_jumps=9, seed=0):
    """Bursty staircase: a few big jumps, the rest tiny."""
    rng = np.random.default_rng(seed)
    edges = np.sort(rng.uniform(0.05, 0.95, n_jumps - 1))
    edges = np.concatenate([[0.0], edges, [1.0]])
    heights = np.cumsum(rng.dirichlet(np.ones(n_jumps) * 0.6))  # spiky
    return np.maximum.accumulate(np.interp(s, edges[1:], heights))

def traj_greedy_mu(s=S):
    # GreedyMu: only μ is gradient-weighted; lv advances uniformly
    return _norm(_greedy_mu_disp(s, seed=1)), _norm(s)

def traj_greedy_joint(s=S):
    # GreedyJoint: both μ AND lv are gradient-weighted → both axes bursty
    mu  = _greedy_mu_disp(s, seed=2)
    lv  = _greedy_mu_disp(s, seed=7)
    return _norm(mu), _norm(lv)


# ── Master method list ────────────────────────────────────────────────────────
#   (label, traj_fn, category, colour, linestyle)
METHODS_TRAJ = [
    # data-agnostic schedules
    ('KL-IG linear',        traj_linear,      'agnostic',    '#1f3b73', '-'),
    ('DDPath cosine',       traj_dd_cosine,   'agnostic',    '#2b6cb0', '-'),
    ('DDPath linear',       traj_dd_linear,   'agnostic',    '#4a90d9', '--'),
    ('DDPath quadratic',    traj_dd_quadratic,'agnostic',    '#6fb1f0', ':'),
    ('PowerPath p=2',       traj_power_p2,    'agnostic',    '#034078', '-.'),
    ('PowerPath p=0.5',     traj_power_p05,   'agnostic',    '#0466c8', '-.'),
    # classifier-informed
    ('KLIG2 pixel',         traj_klig2_pixel, 'classifier',  '#c0392b', '-'),
    ('KL-Descent max-ent',  traj_kld_maxent,  'classifier',  '#e67e22', '-'),
    ('KL-Descent sharp',    traj_kld_sharp,   'classifier',  '#d35400', '--'),
    ('KL-Descent fuzzy',    traj_kld_fuzzy,   'classifier',  '#922b21', ':'),
    # greedy
    ('GreedyMu',            traj_greedy_mu,   'greedy',      '#1e8449', '-'),
    ('GreedyJoint',         traj_greedy_joint,'greedy',      '#27ae60', '--'),
]

CATEGORY_STYLE = {
    'agnostic':   ('Data-agnostic schedules',      '#2b6cb0'),
    'classifier': ('Classifier-informed descent',  '#c0392b'),
    'greedy':     ('Self-referential greedy',       '#1e8449'),
}
print(f'{len(METHODS_TRAJ)} methods across {len(CATEGORY_STYLE)} families ready')

In [ ]:
# ── Figure 1: small multiples — 3×4 grid ─────────────────────────────────────
# 12 panels (11 methods + 1 legend panel).  Linear reference on every panel.

ref_mu, ref_lv = traj_linear()
NCOLS, NROWS = 4, 3
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(13, 10),
                          facecolor='white', sharex=True, sharey=True)
axes_flat = axes.flatten()

for ax_i, (label, fn, cat, colour, ls) in enumerate(METHODS_TRAJ):
    ax = axes_flat[ax_i]
    # faint linear reference
    ax.plot(ref_mu, ref_lv, color='#888', lw=1.0, ls='--', alpha=0.3, zorder=1)
    # method curve
    mu, lv = fn()
    ax.plot(mu, lv, color=colour, lw=2.2, ls=ls, zorder=3)
    ax.scatter([mu[0]],  [lv[0]],  s=30,  color=colour, zorder=5,
                edgecolor='white', lw=0.8)
    ax.scatter([mu[-1]], [lv[-1]], s=55,  color=colour, zorder=5,
                marker='*', edgecolor='white', lw=0.8)
    ax.set_xlim(-0.05, 1.08); ax.set_ylim(-0.05, 1.08)
    ax.set_aspect('equal'); ax.grid(alpha=0.2)
    cat_label, cat_col = CATEGORY_STYLE[cat]
    ax.set_title(label, fontsize=9.5, fontweight='bold', color=colour)
    ax.text(0.03, 0.97, cat_label, transform=ax.transAxes,
             fontsize=7, va='top', color=cat_col,
             bbox=dict(boxstyle='round,pad=0.18', facecolor='white',
                       alpha=0.88, edgecolor='lightgray'))

# last panel → family legend
ax_leg = axes_flat[-1]
ax_leg.axis('off')
handles = []
for cat_key, (cat_label, cat_col) in CATEGORY_STYLE.items():
    handles.append(mlines.Line2D([], [], color=cat_col, lw=2.5,
                                  label=cat_label))
handles.append(mlines.Line2D([], [], color='#888', lw=1.0, ls='--',
                               alpha=0.5, label='Linear (reference)'))
ax_leg.legend(handles=handles, loc='center', fontsize=9,
               frameon=True, edgecolor='lightgray',
               title='Family', title_fontsize=10)

# shared axis labels
for ax in axes[-1, :]: ax.set_xlabel('μ-displacement (norm.)', fontsize=9)
for ax in axes[:, 0]:  ax.set_ylabel('logvar-displacement (norm.)', fontsize=9)

fig.suptitle('Path geometry taxonomy — small multiples (Phase a: synthetic)',
             fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: unified overlay — all 12 curves on one axes ───────────────────
# Three family regions shaded as hull-backgrounds to emphasise clustering.

fig, ax = plt.subplots(figsize=(8, 7), facecolor='white')

# Shaded background hulls (manual bounding boxes per family, for readability)
from matplotlib.patches import FancyBboxPatch
FAMILY_HULLS = {
    # (x0, y0, w, h, colour, alpha, label)
    'agnostic':   (0.00, 0.00, 1.08, 1.08, '#2b6cb0', 0.06),
}
# (we let the curves speak — shading only for the diagonal band)
ax.fill_between([0, 1], [0.12, 1.12], [-0.08, 0.92], alpha=0.06,
                 color='#2b6cb0', zorder=0)

# Linear reference
ref_mu, ref_lv = traj_linear()
ax.plot(ref_mu, ref_lv, color='black', lw=1.2, ls='--', alpha=0.35,
         zorder=2, label='Linear (reference)')

# All methods
for label, fn, cat, colour, ls in METHODS_TRAJ:
    mu, lv = fn()
    ax.plot(mu, lv, color=colour, lw=2.0, ls=ls, zorder=3, label=label)
    ax.scatter([mu[-1]], [lv[-1]], s=50, color=colour, zorder=5,
                marker='*', edgecolor='white', lw=0.8)

# Annotate family cluster regions
cluster_annots = [
    (0.52, 0.58, 'Data-agnostic\nschedules', '#2b6cb0'),
    (0.72, 0.07, 'Classifier-informed\ndescent (pixel only → lv=0)', '#c0392b'),
    (0.38, 0.72, 'Classifier-informed\ndescent (distribution)', '#e67e22'),
    (0.18, 0.45, 'Self-referential\ngreedy', '#1e8449'),
]
for tx, ty, txt, tc in cluster_annots:
    ax.text(tx, ty, txt, fontsize=8, color=tc, ha='center',
             style='italic',
             bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                       alpha=0.82, edgecolor='lightgray'))

ax.set_xlim(-0.04, 1.12); ax.set_ylim(-0.10, 1.12)
ax.set_xlabel('μ-displacement (normalised to [0,1] per method)', fontsize=11)
ax.set_ylabel('logvar-displacement (normalised to [0,1] per method)', fontsize=11)
ax.set_title('All methods: path geometry in (μ, logvar)-displacement space\n'
             '(Phase a — synthetic curves; ★ = endpoint s=1)',
             fontsize=11, fontweight='bold')
ax.grid(alpha=0.2)

# Legend: two columns, grouped by family order
handles, labels_leg = ax.get_legend_handles_labels()
ax.legend(handles, labels_leg, loc='upper left', fontsize=8,
           ncol=2, frameon=True, edgecolor='lightgray',
           framealpha=0.95, columnspacing=0.8, handlelength=2.2)
plt.tight_layout()
plt.show()

### Phase (b) — swap in real paths

Replace each `traj_*` function with one that reads from your computed waypoints:

```python
def traj_real(mu_list, lv_list):
    """mu_list, lv_list: list of tensors (T+1 entries)."""
    mu_seq = np.stack([m.flatten().cpu().numpy() for m in mu_list])  # (T+1, D)
    lv_seq = np.stack([v.flatten().cpu().numpy() for v in lv_list])  # (T+1, D)
    mu_disp = np.linalg.norm(mu_seq - mu_seq[0], axis=1)   # L2 displacement from start
    lv_disp = np.linalg.norm(lv_seq - lv_seq[0], axis=1)
    mu_disp = mu_disp / (mu_disp.max() + 1e-12)
    lv_disp = lv_disp / (lv_disp.max() + 1e-12)
    return mu_disp, lv_disp
```

**Where to get the waypoints per method:**

| Method | Source |
|---|---|
| KL-IG linear, DDPath, PowerPath | `path.at(t, μ_f, lv_f)` over `t = path.steps(T)` — pure math, no model call |
| KL-Descent (any cf) | `path._traj_mu`, `path._traj_lv` — cached after first `at()` |
| GreedyMu / GreedyJoint | `result.waypoints_mu`, `result.waypoints_logvar` |
| KLIG2 pixel-space | only μ moves; set `lv_list = [torch.zeros_like(m) for m in path]` |

Then just pass the `(mu_disp, lv_disp)` tuple into the same `METHODS_TRAJ` list
replacing the synthetic `traj_*` entries and re-run the two figure cells.